# NESCAC DIII Hockey Player Statistics Scraper

This notebook builds a local SQLite database of NESCAC men's hockey player statistics from USCHO conference-stat pages. It is written so that the same scraper can later be pointed at the NESCAC women's hockey pages.

The scraper is intentionally conservative:

- it caches downloaded HTML pages locally so repeated runs do not hammer USCHO;
- it waits between requests;
- it stores both cleaned player-stat tables and raw table snapshots;
- it records which seasons failed, which seasons had no tables, and which rows were dropped for incomplete statistics.

USCHO is a public website, not an API, so the HTML can change. The last section includes diagnostics to help you inspect table names and columns when something breaks.


In [ ]:
# Install requirements if needed. Uncomment in a fresh environment.
#!pip install requests beautifulsoup4 lxml pandas tqdm
#!pip install html5lib beautifulsoup4 lxml



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from __future__ import annotations

import json
import re
import sqlite3
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Optional
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


/workspaces/Decisions-by-Design_v1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

For men, USCHO pages usually follow one of these patterns:

- `https://www.uscho.com/stats/conference/nescac/men-hockey/2025-2026`
- `https://www.uscho.com/stats/conference/nescac/2025-2026`

For women, USCHO pages usually use the conference slug `nescac-women`:

- `https://www.uscho.com/stats/conference/nescac-women/2025-2026`

The scraper tries the candidate URL patterns in order and uses the first successful page with tables.


In [3]:
@dataclass
class ScraperConfig:
    conference_slug: str = "nescac"
    gender_slug: str = "men-hockey"
    start_season: int = 2000
    end_season: int = 2025
    data_dir: Path = Path("data/nescac_hockey_men")
    db_path: Path = Path("data/nescac_hockey_men/nescac_hockey_men.sqlite")

    def season_label(self, start_year):
        return f"{start_year}-{start_year + 1}"

    def season_url(self, start_year):
        season = self.season_label(start_year)
        return (
            f"https://www.uscho.com/stats/conference/"
            f"{self.conference_slug}/{self.gender_slug}/{season}"
        )

MEN_CONFIG = ScraperConfig(
    conference_slug="nescac",
    gender_slug="men-hockey",
    start_season=2000,
    end_season=2025,
)

WOMEN_CONFIG = ScraperConfig(
    conference_slug="nescac",
    gender_slug="women-hockey",
    start_season=2000,
    end_season=2025,
    data_dir=Path("data/nescac_hockey_women"),
    db_path=Path("data/nescac_hockey_women/nescac_hockey_women.sqlite"),
)

CONFIG = MEN_CONFIG
CONFIG


ScraperConfig(conference_slug='nescac', gender_slug='men-hockey', start_season=2000, end_season=2025, data_dir=PosixPath('data/nescac_hockey_men'), db_path=PosixPath('data/nescac_hockey_men/nescac_hockey_men.sqlite'))

In [5]:
test_url = CONFIG.season_url(2025)
print(test_url)

tables = pd.read_html(test_url, flavor="lxml")
print(f"Found {len(tables)} tables")

for i, table in enumerate(tables):
    print()
    print("=" * 60)
    print(f"TABLE {i}")
    print(table.head())

https://www.uscho.com/stats/conference/nescac/men-hockey/2025-2026


ValueError: No tables found matching regex '.+'

## URL helpers and polite downloading


In [5]:
def season_label(start_year: int) -> str:
    return f"{start_year}-{start_year + 1}"


def candidate_urls(config: ScrapeConfig, season: str) -> list[str]:
    base = "https://www.uscho.com/stats/conference/"
    urls = []
    if config.sport_slug:
        urls.append(urljoin(base, f"{config.conference_slug}/{config.sport_slug}/{season}"))
    urls.append(urljoin(base, f"{config.conference_slug}/{season}"))
    return urls


def cache_path(config: ScrapeConfig, url: str) -> Path:
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", url.strip("/"))
    return config.cache_dir / f"{safe}.html"


def fetch_html(url: str, config: ScrapeConfig, force: bool = False) -> tuple[str, str, bool]:
    """Return (html, final_url, from_cache)."""
    config.cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_path(config, url)
    if path.exists() and not force:
        return path.read_text(encoding="utf-8"), url, True

    headers = {"User-Agent": config.user_agent}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    html = response.text
    path.write_text(html, encoding="utf-8")
    time.sleep(config.request_delay_seconds)
    return html, response.url, False


def fetch_first_working_page(config: ScrapeConfig, season: str, force: bool = False) -> dict:
    errors = []
    for url in candidate_urls(config, season):
        try:
            html, final_url, from_cache = fetch_html(url, config, force=force)
            # A successful HTTP page may still be an empty or template page. Require at least one HTML table.
            if "<table" in html.lower():
                return {
                    "season": season,
                    "requested_url": url,
                    "final_url": final_url,
                    "html": html,
                    "from_cache": from_cache,
                    "error": None,
                }
            errors.append(f"{url}: HTTP OK but no <table> tag found")
        except Exception as exc:
            errors.append(f"{url}: {type(exc).__name__}: {exc}")
    return {
        "season": season,
        "requested_url": None,
        "final_url": None,
        "html": None,
        "from_cache": False,
        "error": " | ".join(errors),
    }


## Table extraction

USCHO pages sometimes have several tables on one page. The helper below uses nearby headings to label each table, then `pandas.read_html` to parse the table content.


In [6]:
def normalize_column_name(col) -> str:
    if isinstance(col, tuple):
        parts = [str(x) for x in col if str(x).lower() != "nan"]
        col = "_".join(parts)
    col = str(col).strip()
    col = re.sub(r"\s+", "_", col)
    col = col.replace("%", "pct")
    col = re.sub(r"[^A-Za-z0-9_]+", "", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col.lower() or "unnamed"


def nearby_heading(table_tag) -> str:
    headings = []
    for prev in table_tag.find_all_previous(["h1", "h2", "h3", "h4", "h5", "strong", "caption"], limit=4):
        txt = prev.get_text(" ", strip=True)
        if txt:
            headings.append(txt)
    return " | ".join(reversed(headings)) if headings else "unlabeled_table"


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_column_name(c) for c in df.columns]

    # Drop wholly empty rows and repeated header rows.
    df = df.dropna(how="all")
    if len(df) == 0:
        return df

    first_col = df.columns[0]
    repeated_header = df[first_col].astype(str).str.lower().isin({first_col.lower(), "index", "rank", "player"})
    df = df.loc[~repeated_header].copy()

    # Trim strings and standardize common missing markers.
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "--": pd.NA, "—": pd.NA, "–": pd.NA})
    return df.reset_index(drop=True)


def extract_tables_from_html(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    table_tags = soup.find_all("table")
    parsed_tables = pd.read_html(html)
    records = []
    for i, df in enumerate(parsed_tables):
        heading = nearby_heading(table_tags[i]) if i < len(table_tags) else f"table_{i}"
        clean = clean_dataframe(df)
        if clean.empty:
            continue
        records.append({"table_index": i, "section": heading, "dataframe": clean})
    return records


def infer_table_type(section: str, df: pd.DataFrame) -> str:
    text = f"{section} {' '.join(df.columns)}".lower()
    if any(token in text for token in ["goalie", "goaltend", "gaa", "save", "svpct"]):
        return "goalie"
    if any(token in text for token in ["skater", "scoring", "offense", "defenseman", "freshman"]):
        return "skater"
    if "team" in df.columns and not any(c in df.columns for c in ["player", "name"]):
        return "team"
    return "other"


## Completeness rules

For player tables, this notebook keeps rows that appear to have complete statistics. The default rule is:

- the row has a player/name field, team field, and games-played field when those columns exist;
- all non-metadata statistic columns in that row are non-missing.

You can loosen this if USCHO uses blanks for harmless categories in older seasons.


In [7]:
def find_first_column(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    for c in df.columns:
        if any(cand in c for cand in candidates):
            return c
    return None


def is_player_table(table_type: str, df: pd.DataFrame) -> bool:
    if table_type not in {"skater", "goalie"}:
        return False
    player_col = find_first_column(df, ["player", "name"])
    team_col = find_first_column(df, ["team", "school"])
    return player_col is not None and team_col is not None


def split_complete_rows(df: pd.DataFrame, table_type: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (complete_rows, incomplete_rows)."""
    df = df.copy()
    player_col = find_first_column(df, ["player", "name"])
    team_col = find_first_column(df, ["team", "school"])
    gp_col = find_first_column(df, ["gp", "games_played", "games"])

    required = [c for c in [player_col, team_col, gp_col] if c is not None]
    metadata = {"season", "source_url", "table_index", "section", "table_type", "fetched_at", "requested_url", "final_url"}
    optional = {"index", "rank", "rk", "no", "number"}
    stat_cols = [c for c in df.columns if c not in metadata and c not in optional]
    required_for_completeness = sorted(set(required + stat_cols))

    if not required_for_completeness:
        return df.iloc[0:0].copy(), df.copy()

    complete_mask = df[required_for_completeness].notna().all(axis=1)
    complete = df.loc[complete_mask].copy()
    incomplete = df.loc[~complete_mask].copy()
    return complete, incomplete


## SQLite writer

The database stores:

- `season_log`: one row per attempted season;
- `raw_tables`: JSON snapshots of every parsed table;
- `player_stats_complete`: complete skater/goalie player-stat rows;
- `player_stats_incomplete`: player-stat rows that were dropped by the completeness rule;
- `non_player_tables`: team/other tables, useful for diagnostics or later feature engineering.

Because USCHO columns can vary by season and table type, the cleaned stat tables are saved in a flexible wide format. SQLite will receive the union of all columns found in the run.


In [23]:
def write_table(conn: sqlite3.Connection, name: str, df: pd.DataFrame, if_exists: str = "replace") -> None:
    if df.empty:
        # Create an empty marker table with a simple schema.
        pd.DataFrame({"_empty": []}).to_sql(name, conn, if_exists=if_exists, index=False)
    else:
        df.to_sql(name, conn, if_exists=if_exists, index=False)


def build_database(config: ScrapeConfig, force_refresh: bool = False) -> dict[str, pd.DataFrame]:
    seasons = [season_label(y) for y in range(config.first_start_year, config.last_start_year + 1)]
    season_logs = []
    raw_tables = []
    complete_players = []
    incomplete_players = []
    non_player_tables = []

    fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

    for season in tqdm(seasons, desc="Scraping seasons"):
        page = fetch_first_working_page(config, season, force=force_refresh)
        season_logs.append({k: v for k, v in page.items() if k != "html"})
        if page["html"] is None:
            continue

        tables = extract_tables_from_html(page["html"])
        for t in tables:
            df = t["dataframe"].copy()
            table_type = infer_table_type(t["section"], df)
            df.insert(0, "season", season)
            df.insert(1, "table_type", table_type)
            df.insert(2, "section", t["section"])
            df.insert(3, "table_index", t["table_index"])
            df.insert(4, "source_url", page["final_url"])
            df.insert(5, "fetched_at", fetched_at)

            raw_tables.append({
                "season": season,
                "table_index": t["table_index"],
                "section": t["section"],
                "table_type": table_type,
                "source_url": page["final_url"],
                "fetched_at": fetched_at,
                "table_json": df.to_json(orient="records"),
            })

            if is_player_table(table_type, df):
                complete, incomplete = split_complete_rows(df, table_type)
                if not complete.empty:
                    complete_players.append(complete)
                if not incomplete.empty:
                    incomplete_players.append(incomplete)
            else:
                non_player_tables.append(df)

    outputs = {
        "season_log": pd.DataFrame(season_logs),
        "raw_tables": pd.DataFrame(raw_tables),
        "player_stats_complete": pd.concat(complete_players, ignore_index=True, sort=False) if complete_players else pd.DataFrame(),
        "player_stats_incomplete": pd.concat(incomplete_players, ignore_index=True, sort=False) if incomplete_players else pd.DataFrame(),
        "non_player_tables": pd.concat(non_player_tables, ignore_index=True, sort=False) if non_player_tables else pd.DataFrame(),
    }

    with sqlite3.connect(config.db_path) as conn:
        for name, df in outputs.items():
            write_table(conn, name, df)
        # Check table schema before creating indexes

            columns = {
                row[1]
                for row in conn.execute("PRAGMA table_info(player_stats_complete)")
            }

            print("player_stats_complete columns:")
            print(sorted(columns))

            if "season" in columns:
                conn.execute("""
                    CREATE INDEX IF NOT EXISTS
                    idx_player_complete_season
                    ON player_stats_complete(season)
                """)
            else:
                print("Skipping season index; no 'season' column found.")

            if "table_type" in columns:
                conn.execute("""
                    CREATE INDEX IF NOT EXISTS
                    idx_player_complete_type
                    ON player_stats_complete(table_type)
                """)
            else:
                print("Skipping table_type index; no 'table_type' column found.")

    return outputs


## Build the men's database

The first run may take a few minutes because it downloads and caches all seasons. Later runs are much faster unless you pass `force_refresh=True`.


In [24]:
outputs = build_database(CONFIG, force_refresh=False)

for name, df in outputs.items():
    print(f"{name:25s} {len(df):6d} rows")

#print(f"
#Wrote SQLite database to: {CONFIG.db_path.resolve()}")


Scraping seasons: 100%|██████████| 26/26 [00:56<00:00,  2.18s/it]

player_stats_complete columns:
['_empty']
Skipping season index; no 'season' column found.
Skipping table_type index; no 'table_type' column found.
player_stats_complete columns:
['_empty']
Skipping season index; no 'season' column found.
Skipping table_type index; no 'table_type' column found.
player_stats_complete columns:
['_empty']
Skipping season index; no 'season' column found.
Skipping table_type index; no 'table_type' column found.
player_stats_complete columns:
['_empty']
Skipping season index; no 'season' column found.
Skipping table_type index; no 'table_type' column found.
player_stats_complete columns:
['_empty']
Skipping season index; no 'season' column found.
Skipping table_type index; no 'table_type' column found.
season_log                    26 rows
raw_tables                     0 rows
player_stats_complete          0 rows
player_stats_incomplete        0 rows
non_player_tables              0 rows


In [22]:
# Preview complete player statistics.
outputs["player_stats_complete"].head(10)


""


In [ ]:
# Which seasons failed or had no usable tables?
log = outputs["season_log"].copy()
log.loc[log["error"].notna() & (log["error"] != ""), ["season", "error"]].head(20)


## Inspect table sections and columns

Run this when a page parses but the player table is not classified correctly. It shows the table sections and normalized column names found by the scraper.


In [ ]:
def inspect_cached_season(config: ScrapeConfig, season: str) -> pd.DataFrame:
    page = fetch_first_working_page(config, season, force=False)
    if page["html"] is None:
        raise RuntimeError(page["error"])
    rows = []
    for t in extract_tables_from_html(page["html"]):
        df = t["dataframe"]
        rows.append({
            "season": season,
            "table_index": t["table_index"],
            "section": t["section"],
            "inferred_type": infer_table_type(t["section"], df),
            "columns": ", ".join(df.columns),
            "rows": len(df),
        })
    return pd.DataFrame(rows)

inspect_cached_season(CONFIG, season_label(CONFIG.last_start_year))


## Query examples


In [ ]:
with sqlite3.connect(CONFIG.db_path) as conn:
    # Example: top complete skater rows by points, if a pts-like column exists.
    df = pd.read_sql("SELECT * FROM player_stats_complete LIMIT 5", conn)

df


In [ ]:
def show_columns(db_path: Path, table: str = "player_stats_complete") -> list[str]:
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql(f"PRAGMA table_info({table})", conn)["name"].tolist()

show_columns(CONFIG.db_path)


In [ ]:
def query_top_points(db_path: Path, season: Optional[str] = None, limit: int = 25) -> pd.DataFrame:
    cols = show_columns(db_path, "player_stats_complete")
    pts_col = next((c for c in cols if c in {"pts", "points", "p"}), None)
    player_col = next((c for c in cols if c in {"player", "name"} or "player" in c), None)
    team_col = next((c for c in cols if c in {"team", "school"} or "team" in c), None)
    if not all([pts_col, player_col, team_col]):
        raise ValueError("Could not identify player/team/points columns. Inspect columns with show_columns().")

    where = "WHERE table_type = 'skater'"
    params = []
    if season:
        where += " AND season = ?"
        params.append(season)

    sql = f"""
        SELECT season, {player_col} AS player, {team_col} AS team, {pts_col} AS points, *
        FROM player_stats_complete
        {where}
        ORDER BY CAST({pts_col} AS REAL) DESC
        LIMIT ?
    """
    params.append(limit)
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql(sql, conn, params=params)

# query_top_points(CONFIG.db_path, season="2024-2025", limit=10)


## Build the women's database later

When you are ready to build the women's database, switch `CONFIG` to `WOMEN_CONFIG` and re-run the build cell.


In [ ]:
# CONFIG = WOMEN_CONFIG
# outputs_women = build_database(CONFIG, force_refresh=False)
# print(f"Wrote SQLite database to: {CONFIG.db_path.resolve()}")


## Notes for maintenance

- If USCHO changes table headings or column names, adjust `infer_table_type` and the column candidates in `split_complete_rows`.
- If older seasons use blanks where modern seasons use zeroes, loosen `split_complete_rows` so it only requires identity columns plus the core statistics you care about.
- Keep the HTML cache if you need reproducibility. Delete `uscho_cache/` or use `force_refresh=True` to refresh everything.
- Consider committing only the notebook and not the cache/database if the data are regenerated regularly.
